# Preserve mode 1 with three launch amplitudes

Run the same three-mode fiber at **100 kW** and **81.943 kW**. Each bounded search
updates all three real launch coefficients and minimizes distortion of **mode 1 (LP01)**
between its original input and nonlinear output:

$$F_{\rm IO}=\frac{|\int A_1^*(0,t)A_{{\rm NL},1}(L,t)\,dt|^2}
{(\int|A_1(0,t)|^2dt)(\int|A_{{\rm NL},1}(L,t)|^2dt)},
\qquad D_{\rm IO}=1-F_{\rm IO}.$$

Both normalization energies belong to mode 1. All three modes still propagate and
interact, so changing modes 2 and 3 can influence the mode-1 output. The fields use
the same time grid with **no temporal alignment**. Dispersion and nonlinear reshaping
both contribute. A common overall phase or amplitude change does not change this
normalized score; the separate LP01 photon floor retains useful mode-1 brightness.

Use $0\le c_1,c_2,c_3\le100$, with $c_1+c_2+c_3=100$. **All three coefficients are
adjustable.** Peak power is fixed within each case. The pulse constructor normalizes
amplitudes, so $P_m/P=c_m^2/\sum_jc_j^2$; coefficients are not power percentages.
All launch phase differences remain zero.

Finite-difference projected gradient descent uses up to 20 accepted iterations.
One 1 mm baseline/selected check accompanies the 5 mm search. Linear propagation is
computed only for displayed reference curves. Run all cells in order with
**GNLSE (Julia)** after changing settings.

In [ ]:
using Pkg
workspace = let dir = abspath(@__DIR__)
    while !isfile(joinpath(dir, "notebook_helpers", "run_library.py"))
        dirname(dir) == dir && error("Open this notebook inside the Nonlinear Optics workspace.")
        dir = dirname(dir)
    end
    dir
end
ROOT = get(ENV, "DIO_PROJECT_ROOT", joinpath(workspace, "FastDifferentiableMMNLO"))
Pkg.activate(ROOT)
using PulsePropagation, FFTW, LinearAlgebra, Printf, TOML, Dates, SHA, NPZ, Serialization
using PyPlot

BLAS.set_num_threads(1)
FFTW.set_num_threads(1)
run_dir = haskey(ENV, "DIO_RUN_DIR") ? ENV["DIO_RUN_DIR"] :
    mktempdir(mkpath(joinpath(workspace, "output", "organized", "runs", "Modal amplitude coefficients", "Gradient descent")); prefix=Dates.format(now(), "yyyy-mm-dd") * "_dio_three_coefficient_optimization_", cleanup=false)
mkpath(run_dir)
println("Results: ", run_dir)

# Load the launch, objective, descent, and plotting functions used below.
include(joinpath(workspace, "notebook_helpers", "julia", "Modal amplitude coefficients", "dio_three_coefficient_optimization.jl"))

## 1. Controls, normalization, and budgets

Start both power cases at equal amplitudes. Internally use $u=c/100$ and project each
proposal onto $u_m\ge0$, $\sum_m u_m=1$. All nonnegative, equal-phase modal mixtures
are accessible without duplicate overall coefficient scalings.

The **0.5 coefficient perturbation** estimates the gradient. The learning rate and
backtracking set the actual move, capped at 10 per coefficient. Each central gradient
typically needs six nonlinear propagations. The original **20-iteration** and
**200-optimization-evaluation** budgets are retained for each fixed power.

Validation only evaluates the coarse baseline, its 1 mm counterpart, and the selected
launch at 1 mm (at most four validation evaluations per power). Cached repeats are free.
The triangle plots show existing evaluations and do not trigger a sweep.

In [ ]:
power_cases_W = [100_000.0, 81_943.0]
start_coefficients = fill(100.0/3, 3)
coefficient_bounds = (0.0, 100.0)
fd_step = 0.5                       # amplitude-coefficient units
learning_rate = 0.10                # update u = c/100
max_iterations = 20
max_optimizer_evaluations = 200     # per fixed-power case
max_check_evaluations = 4           # baseline coarse/fine + selected fine, per case
max_coefficient_change = 10.0
gradient_tolerance = 1e-4           # infinity norm of projected gradient in u
minimum_brightness_fraction = 0.50
search_dz = 0.005                   # 5 mm
validation_dz = 0.001               # 1 mm
dio_tolerance = 2e-4
minimum_decrease = 1e-10

settings = Dict(
    "control"=>"three nonnegative real amplitudes; sum(c)=100; zero relative phases",
    "objective"=>"D_IO = 1 - normalized squared complex overlap of original input and nonlinear output, mode 1 (LP01) only",
    "temporal_alignment"=>"none; identical original time grid",
    "objective_modes"=>[1], "objective_filter"=>"none",
    "power_cases_W"=>power_cases_W, "start_coefficients"=>start_coefficients,
    "coefficient_bounds"=>collect(coefficient_bounds),
    "finite_difference_step_coefficient"=>fd_step,
    "learning_rate_u"=>learning_rate, "max_iterations"=>max_iterations,
    "max_optimizer_evaluations_per_power"=>max_optimizer_evaluations,
    "max_check_evaluations_per_power"=>max_check_evaluations,
    "max_coefficient_change"=>max_coefficient_change,
    "gradient_tolerance_u"=>gradient_tolerance,
    "minimum_brightness_fraction"=>minimum_brightness_fraction,
    "brightness_modes"=>[1],
    "brightness_definition"=>"LP01 unfiltered photons relative to equal-three-mode fine baseline at the same power",
    "search_dz_m"=>search_dz, "validation_dz_m"=>validation_dz,
    "dio_tolerance"=>dio_tolerance,
    "armijo_c1"=>1e-4,"max_backtracking_halvings"=>12,
    "minimum_D_IO_decrease"=>minimum_decrease,
    "Nt"=>2048, "time_window_ps"=>10.0,
    "pulse_fwhm_ps"=>0.2, "fiber_length_m"=>2.86, "carrier_m"=>1379.2e-9,
    "core_radius_m"=>52.5e-6, "NA"=>0.1, "num_modes"=>3,
    "linear_reference"=>"diagnostic plots only: same candidate input and fiber, n2 = 0",
    "cp_target_error"=>1e-3, "cp_seed"=>Int(0x4d4d474e),
    "julia_version"=>string(VERSION),
    "manifest_sha256"=>bytes2hex(sha256(read(joinpath(ROOT, "Manifest.toml")))),
)
for P in power_cases_W
    mkpath(case_dir(P))
end
write_toml(joinpath(run_dir, "settings.toml"), settings)
println("Fixed powers (kW): ", power_cases_W./1000)
println("Initial coefficients: ", start_coefficients)

## 2. Build the same fiber and CP representation once

Mode 1 is LP01; modes 2 and 3 are the cosine and sine LP11 orientations. The fiber,
dispersion and CP overlap representation are reused for both fixed-power searches.
The objective and brightness floor use mode 1 (LP01) only. All three modes propagate. Linear propagation is
reserved for baseline/selected diagnostic plots and is outside the objective.

In [ ]:
properties = compute_fiber_properties(
    StepIndex(settings["carrier_m"], settings["core_radius_m"], settings["NA"]),
    Silica(), settings["num_modes"];
    polarization=:scalar, mode_basis=:LP, beta_order=5)
S_cp = cp_compress(properties.S; error=settings["cp_target_error"], seed=settings["cp_seed"])
context_cache = Dict{Tuple{Int,Symbol},Any}()
main_context = simulation_context(settings["Nt"])
time_grid, domain = main_context.grid, main_context.domain
mode_names = ("LP01", "LP11-cos", "LP11-sin")
@printf("CP rank: %d; relative tensor error: %.4g\n", cp_rank(S_cp), S_cp.relative_error)
println("Mode labels: ", properties.modes.labels)

## 3. Evaluate the mode-1 input/output overlap

`dio_at(c, P)` normalizes the amplitude vector and propagates all three modes through
the nonlinear fiber. It compares only the original and output LP01 fields, without
shifting either in time. Every new attempted propagation is counted before solving;
cached repeats are free. Launch power and modal fractions are checked per evaluation.

The all-zero launch is invalid. If LP01 is absent at input or output, its normalized
overlap is undefined: store `F_IO=NaN`, use a worst-score penalty `D_IO=1` for probes,
and reject that trial as an accepted solution. The 50% LP01 brightness floor is separate.

In [ ]:
evaluation_cache = Dict{Any,NamedTuple}()
field_cache = Dict{Int,NamedTuple}()
evaluations, failures = NamedTuple[], NamedTuple[]
attempt_id = Ref(0)
used = Dict(P=>Dict(:optimization=>0, :validation=>0) for P in power_cases_W)

# Cheap checks of the constraints and the LP01-only overlap definition.
let
    for v in ([0.2,0.3,0.5],[-2.0,3.0,0.4],[1.0,1.0,1.0])
        u=project_simplex(v)
        @assert all(u .>= 0) && isapprox(sum(u),1.0;atol=1e-14)
        @assert isapprox(project_simplex(u),u;atol=1e-14)
    end
    @assert canonical_coefficients([20,30,50]) ≈ canonical_coefficients([40,60,100])
    a=zeros(ComplexF64,4,3,1); a[1,1,1]=1; a[1,2,1]=1
    b=copy(a); b[1,3,1]=2
    @assert isapprox(field_fidelity(a,b),1;atol=1e-12)
    distorted=copy(a); distorted[2,1,1]=1
    @assert field_fidelity(a,distorted)<1
    empty=copy(a); empty[:,1,:].=0
    @assert isnan(field_fidelity(empty,b;allow_empty=true))
    @assert isapprox(field_fidelity(a,2exp(0.7im).*a),1;atol=1e-12)
    for m in 1:3
        pure=zero(a); pure[1,m,1]=1
        @assert field_fidelity(pure,pure;selected_modes=m:m)==1
    end
end
println("Projection, scale-equivalence, and LP01-only overlap checks passed.")

# Linear propagation is attached only to the baseline/selected fields used in figures.
linear_diagnostics = NamedTuple[]

## 4. Estimate a three-component gradient

For component $i$, change only raw $c_i$ by $\pm0.5$, then normalize the complete
launch. Each probe uses the same fixed total peak power and one nonlinear propagation.
Multiplying $dD_{\rm IO}/dc_i$ by 100 gives $dD_{\rm IO}/du_i$.

Probe vectors need not sum to 100 before normalization. Accepted states are projected
onto the sum-100 triangle. A second-order one-sided stencil handles the boundaries.

In [ ]:
# The setup cell loads these functions from the study helper file.

## 5. Establish the baseline and brightness floor

Evaluate the equal-amplitude baseline once at 5 mm and once at 1 mm. The 1 mm
LP01 photon count defines the 50% brightness floor for that fixed power.
Record the objective difference; the final result reports whether it meets tolerance.

In [ ]:
preflights=Dict{Float64,Any}()
brightness_floors=Dict{Float64,Float64}()
for P in power_cases_W
    baseline=dio_at(start_coefficients,P)
    fine=dio_at(start_coefficients,P;dz=validation_dz)
    step_error=abs(baseline.D_IO-fine.D_IO)
    ready=step_error<=dio_tolerance
    brightness_floors[P]=minimum_brightness_fraction*fine.photons
    preflights[P]=(;baseline,fine,step_error,ready)
    write_toml(joinpath(case_dir(P),"baseline_check.toml"),Dict(
        "power_W"=>P,"step_size_agreement"=>ready,"baseline_D_IO_search"=>baseline.D_IO,
        "baseline_D_IO_fine"=>fine.D_IO,"step_error"=>step_error,
        "brightness_floor"=>brightness_floors[P]))
    @printf("P=%.3f kW: baseline D_IO %.8f (5 mm), %.8f (1 mm); difference %.3g; agreement=%s\n",
        P/1000,baseline.D_IO,fine.D_IO,step_error,string(ready))
end

## 6. Update the full vector with projected gradient descent

At the current launch, estimate all three derivatives. Propose
$v=\Pi(u-0.10\nabla D)$, where $\Pi$ projects onto the nonnegative sum-one triangle.
Cap the largest coefficient move at 10. Backtracking repeatedly halves this move
until D_IO decreases sufficiently and the LP01 brightness floor is met.

Stop at 20 accepted iterations, the 200-evaluation budget, a small projected gradient,
or an unsuccessful line search. A stop reason is not by itself evidence of a minimum;
the next section checks the selected point on the finer grid.

In [ ]:
optimizations=Dict(P=>optimize_launch(P) for P in power_cases_W)

## 7. Check the selected launch at 1 mm

Re-evaluate the selected launch once at 1 mm and compare its objective with the saved
1 mm baseline. Check coefficients, accepted objective decreases, the step cap and
brightness. The 5 mm/1 mm differences are reported against the original tolerance.

No extra gradient probes, nearby points, dense-tensor comparisons, time-grid doubling,
or previous-optimum campaigns are run. A bounded-search stop is reported as observed;
improvement and convergence are not assumed.

In [ ]:
validations=Dict{Float64,Any}()
summaries=Dict{Float64,Any}()
comparison_rows=NamedTuple[]
for P in power_cases_W
    opt=optimizations[P]
    base_fine=preflights[P].fine
    c=coefficients(opt.best)
    best_fine=dio_at(c,P;dz=validation_dz)
    dz_error=max(preflights[P].step_error,abs(opt.best.D_IO-best_fine.D_IO))
    coefficients_ok=all(r->all(x->0<=x<=100,coefficients(r)) &&
        isapprox(sum(coefficients(r)),100;atol=1e-10),opt.history)
    decreasing=all(diff([r.D_IO for r in opt.history]).<0)
    steps_ok=all(r->r.max_coefficient_step<=max_coefficient_change+1e-10,opt.history)
    path_brightness=all(r->r.brightness_ratio>=minimum_brightness_fraction,opt.history)
    checks=Dict(
        "overlap_and_launch_sanity"=>true,
        "accepted_coefficients_feasible"=>coefficients_ok,
        "accepted_objective_decreases"=>decreasing,
        "accepted_step_cap"=>steps_ok,
        "accepted_path_brightness"=>path_brightness,
        "baseline_selected_step_size_agreement"=>dz_error<=dio_tolerance,
        "fine_brightness"=>best_fine.photons>=brightness_floors[P],
        "fine_objective_defined"=>base_fine.objective_defined && best_fine.objective_defined)
    core_ok=all(values(checks))
    improvement=base_fine.D_IO-best_fine.D_IO
    verified_improvement=core_ok && improvement>dio_tolerance
    summary=Dict(
        "power_W"=>P,"objective"=>"mode-1 original-input versus nonlinear-output D_IO",
        "objective_modes"=>[1],"brightness_modes"=>[1],"temporal_alignment"=>"none",
        "stop_reason"=>opt.status,"accepted_steps"=>length(opt.history)-1,
        "selected_coefficients"=>c,
        "selected_input_power_fractions"=>c.^2 ./ sum(abs2,c),
        "baseline_D_IO_fine"=>base_fine.D_IO,"selected_D_IO_fine"=>best_fine.D_IO,
        "baseline_D_IO_search"=>preflights[P].baseline.D_IO,"selected_D_IO_search"=>opt.best.D_IO,
        "absolute_D_IO_improvement"=>improvement,
        "fractional_D_IO_reduction"=>improvement/base_fine.D_IO,
        "brightness_ratio"=>best_fine.photons/base_fine.photons,
        "max_step_size_D_IO_difference"=>dz_error,
        "core_checks_passed"=>core_ok,"verified_improvement"=>verified_improvement,
        "fine_grid_stationarity_checked"=>false,"global_minimum_established"=>false,
        "new_optimization_evaluations"=>used[P][:optimization],
        "new_check_evaluations"=>used[P][:validation],
        "failed_evaluations"=>count(r->r.power_W==P,failures),
        "checks"=>Dict(k=>(v ? "pass" : "fail") for (k,v) in checks))
    summaries[P]=summary
    validations[P]=(;base_fine,best_fine)
    for (label,row) in (("equal_three_mode_baseline",base_fine),("three_coefficient_selected",best_fine))
        push!(comparison_rows,comparison_row(label,row,base_fine))
        attach_linear_diagnostic!(row)
    end
    write_toml(joinpath(case_dir(P),"summary.toml"),summary)
    save_rows(joinpath(case_dir(P),"comparisons.csv"),filter(r->r.power_W==P,comparison_rows))
    @printf("%.3f kW: fine D_IO %.8f -> %.8f; core checks=%s; stop=%s\n",
        P/1000,base_fine.D_IO,best_fine.D_IO,string(core_ok),opt.status)
    @printf("  selected %s; optimization %d/%d, checks %d/%d\n",coeff_label(c),
        used[P][:optimization],max_optimizer_evaluations,used[P][:validation],max_check_evaluations)
end
save_rows(joinpath(run_dir,"comparisons.csv"),comparison_rows)
write_toml(joinpath(run_dir,"summary.toml"),Dict(power_label(P)=>summaries[P] for P in power_cases_W))
# Saved state supports figure-only updates of the shared scale without another simulation.
serialize(joinpath(run_dir,"plot_state.jls"),Dict(
    "properties"=>properties,"time_grid"=>time_grid,"optimizations"=>optimizations,
    "validations"=>validations,"summaries"=>summaries,"evaluations"=>evaluations,
    "field_cache"=>field_cache,"used"=>used,"mode_names"=>mode_names))

## 8. Watch the search move through the three-mode mixtures

Each row is one fixed power. Read left to right: the score, all three amplitude
coefficients, and the corresponding **input power fractions**. The triangle below
places pure LP01, LP11-cos, and LP11-sin at its corners. Dots are evaluations that
were actually computed; the line joins accepted steps. This is not a surface fit or sweep.

In [ ]:
mode_colors=("#366b9e","#cf7c32","#439576")
mode_styles=("-","-","--")
mode_markers=("o","s","none")
fig,axes=subplots(2,3;figsize=(16,8.4),constrained_layout=true)
for (row_index,P) in enumerate(power_cases_W)
    opt=optimizations[P];h=opt.history
    iterations=[r.iteration for r in h]
    axes[row_index,1].plot(iterations,[r.D_IO for r in h],"o-";
        color=mode_colors[1],markersize=4,label="Accepted path: 5 mm")
    axes[row_index,1].scatter([0,iterations[end]],
        [validations[P].base_fine.D_IO,validations[P].best_fine.D_IO];
        color="black",marker="x",s=60,label="Fine check: 1 mm",zorder=5)
    axes[row_index,1].set_ylabel("LP01 D_IO")
    axes[row_index,1].set_title(@sprintf("%.3f kW | %s",P/1000,replace(opt.status,"_"=>" "));fontsize=10)
    axes[row_index,1].legend(;fontsize=8)
    for m in 1:3
        axes[row_index,2].plot(iterations,[coefficients(r)[m] for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,label=mode_names[m])
        axes[row_index,3].plot(iterations,[100coefficients(r)[m]^2/sum(abs2,coefficients(r)) for r in h];
            color=mode_colors[m],linestyle=mode_styles[m],marker=mode_markers[m],markersize=3,label=mode_names[m])
    end
    axes[row_index,2].set_ylabel("Amplitude coefficient (sum = 100)")
    axes[row_index,3].set_ylabel("Input modal power (%)")
    for j in 2:3
        axes[row_index,j].set_ylim(-2,102)
        axes[row_index,j].legend(;fontsize=8)
    end
    for j in 1:3
        axes[row_index,j].set_xlabel("Accepted iteration")
        tidy_axis(axes[row_index,j])
    end
end
fig.suptitle("Three adjustable launch amplitudes; bounded search results")
finish_figure(fig,joinpath(run_dir,"optimization_history"))
fig,axes=subplots(1,2;figsize=(13,5.7),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    ax=axes[j]
    rows=filter(r->r.power_W==P && r.dz==search_dz && r.nt==settings["Nt"] && r.representation=="cp",evaluations)
    xy=[triangle_xy(coefficients(r)) for r in rows]
    dots=ax.scatter(first.(xy),last.(xy);c=[r.D_IO for r in rows],cmap="viridis",s=18,alpha=0.8)
    path=[triangle_xy(coefficients(r)) for r in optimizations[P].history]
    ax.plot(first.(path),last.(path),"o-";color="#ca584b",markersize=3,label="Accepted steps",zorder=5)
    ax.scatter([first(path)[1]],[first(path)[2]];marker="o",s=85,
        facecolors="white",edgecolors="black",label="Equal-amplitude start",zorder=6)
    ax.scatter([last(path)[1]],[last(path)[2]];marker="*",s=155,
        color="#e79b36",edgecolors="black",label="Selected",zorder=7)
    ax.plot([0,1,0.5,0],[0,0,sqrt(3)/2,0];color="#888888",linewidth=1)
    ax.text(-0.03,-0.035,"LP01";ha="center",fontsize=10)
    ax.text(1.03,-0.035,"LP11-cos";ha="center",fontsize=10)
    ax.text(0.5,sqrt(3)/2+0.025,"LP11-sin";ha="center",fontsize=10)
    ax.set_aspect("equal");ax.set_xlim(-0.13,1.13);ax.set_ylim(-0.12,1.0)
    ax.axis("off")
    ax.set_title(@sprintf("%.3f kW: mixture by input power fraction",P/1000))
    ax.legend(;loc="upper left",fontsize=7.5)
    fig.colorbar(dots;ax,shrink=0.65,label="Sampled LP01 D_IO (5 mm)")
end
finish_figure(fig,joinpath(run_dir,"sampled_mixture_triangles"))

## 9. Compare baseline and selected output pulses

Both comparisons use the 1 mm propagation step and the original-input/nonlinear-output
objective in mode 1 only. All three modes remain visible in the diagnostics. At each launch, display the original input, a linear
diagnostic reference, and the nonlinear output in time and wavelength. No temporal
shift is applied. Spectra keep their frequency-bin weights on a wavelength axis.

In [ ]:
fig,axes=subplots(1,2;figsize=(13,4.8),constrained_layout=true)
for (j,P) in enumerate(power_cases_W)
    v=validations[P]
    rows=(v.base_fine,v.best_fine)
    labels=["Equal three\nmode start","Three coefficients\nselected"]
    bars=axes[j].bar(1:2,[r.D_IO for r in rows];color=["#a8b6c6","#366b9e"])
    for k in 1:2
        axes[j].text(k,rows[k].D_IO+0.012,@sprintf("%.4f",rows[k].D_IO);ha="center",fontsize=9)
    end
    axes[j].set_xticks(1:2,labels;fontsize=8)
    axes[j].set_ylim(0,min(1.05,1.17maximum(r.D_IO for r in rows)+0.02))
    axes[j].set_ylabel("LP01 D_IO (1 mm)")
    axes[j].set_title(@sprintf("%.3f kW",P/1000));tidy_axis(axes[j])
end
finish_figure(fig,joinpath(run_dir,"launch_comparisons"))

times_ps=collect(time_axis(time_grid))
wavelengths_nm=collect(wavelength_axis(time_grid,properties.omega0)).*1e9
wavelength_indices=findall(isfinite.(wavelengths_nm).&(wavelengths_nm.>0))
wavelength_indices=wavelength_indices[sortperm(wavelengths_nm[wavelength_indices])]
curve_labels=("Input","Linear reference","Nonlinear output")
curve_colors=("#777777","#cf7c32","#366b9e")
curve_styles=(":","--","-")
for P in power_cases_W
    plot_modal_output("baseline",validations[P].base_fine)
    plot_modal_output("selected",validations[P].best_fine)
end

## 10. See the spatial modes and where the energy goes

The spatial mode shapes are fixed by the fiber. Their signed field amplitudes are
shown below. The bars show how the total pulse energy is distributed among modes;
these fractions can change during nonlinear propagation.

In [ ]:
x,y=properties.modes.x,properties.modes.y
modes=properties.modes.fields # y, x, mode
let fig=figure(;figsize=(15,4.8),constrained_layout=true)
    for m in 1:3
        ax=subplot(1,3,m)
        profile=real.(modes[:,:,m]);profile ./= maximum(abs,profile)
        mesh=ax.pcolormesh(x.*1e6,y.*1e6,profile;cmap="seismic",vmin=-1,vmax=1,
            shading="auto",rasterized=true)
        theta=range(0,2pi;length=181)
        ax.plot(settings["core_radius_m"].*1e6.*cos.(theta),settings["core_radius_m"].*1e6.*sin.(theta);
            color="#555555",linestyle="--",linewidth=0.8)
        ax.set_title("Mode $m: "*mode_names[m]);ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)")
        ax.set_aspect("equal")
        fig.colorbar(mesh;ax,shrink=0.75,label="Relative field amplitude")
    end
    fig.suptitle("Fixed spatial modes; dashed circle marks the fiber core")
    finish_figure(fig,joinpath(run_dir,"spatial_modes"))
end
npzwrite(joinpath(run_dir,"spatial_mode_fields.npz"),Dict("x_m"=>x,"y_m"=>y,"modes_y_x_m"=>modes))
modal_rows=NamedTuple[]
for P in power_cases_W
    fig,axes=subplots(1,2;figsize=(12.5,4.5),constrained_layout=true,sharey=true)
    for (case_index,(label,row)) in enumerate((("Baseline",validations[P].base_fine),("Selected",validations[P].best_fine)))
        saved=field_cache[row.id]
        for (j,a) in enumerate((saved.input,saved.linear,saved.nonlinear))
            energy=vec(sum(abs2.(a);dims=(1,3)));fractions=energy./sum(energy)
            axes[case_index].bar(collect(1:3).+(j-2)*0.23,100fractions;
                width=0.23,color=curve_colors[j],label=curve_labels[j])
            for m in 1:3
                push!(modal_rows,(;power_W=P,case=lowercase(label),branch=curve_labels[j],mode=m,energy_fraction=fractions[m]))
            end
        end
        axes[case_index].set_xticks(1:3,collect(mode_names))
        axes[case_index].set_ylim(0,105);axes[case_index].set_ylabel("Fraction of total pulse energy (%)")
        axes[case_index].set_title(label*" "*coeff_label(coefficients(row));fontsize=10)
        axes[case_index].legend(;fontsize=8);tidy_axis(axes[case_index])
    end
    fig.suptitle(@sprintf("%.3f kW: modal energy fractions",P/1000))
    finish_figure(fig,joinpath(case_dir(P),"modal_energy_fractions"))
end
save_rows(joinpath(run_dir,"modal_energy_fractions.csv"),modal_rows)

## 11. Reconstruct the whole beam, including interference

At each branch's total pulse peak, add the complex modal fields before squaring.
The spatial integral is checked against the summed modal peak power. Every physical
beam and single-mode intensity image reads the same absolute scale from
`plot_settings.toml` beside this notebook. Signed mode basis maps retain −1 to 1.

Individual nonlinear-mode maps use each mode's own peak time and therefore cannot
be summed to reconstruct the coherent whole-beam map.

In [ ]:
beam_maps=Dict{Tuple{Float64,String},Any}()
individual_maps=NamedTuple[]
for P in power_cases_W, (case_index,(label,row)) in enumerate((("baseline",validations[P].base_fine),("selected",validations[P].best_fine)))
    saved=field_cache[row.id]
    beam_maps[(P,label)]=[peak_beam_map(a) for a in (saved.input,saved.linear,saved.nonlinear)]
    fractions=[row.mode1_energy_fraction,row.mode2_energy_fraction,row.mode3_energy_fraction]
    for m in 1:3
        power_t=vec(sum(abs2.(saved.nonlinear[:,m,:]);dims=2));k=argmax(power_t)
        intensity=abs2.(modes[:,:,m]).*power_t[k]./1e12
        push!(individual_maps,(;power_W=P,case_index,label,mode=m,intensity,
            peak_power_W=power_t[k],time_ps=times_ps[k],energy_fraction=fractions[m]))
    end
end
observed_max=max(maximum(maximum(m.intensity) for maps in values(beam_maps) for m in maps),
    maximum(maximum(m.intensity) for m in individual_maps))
shared_plot_settings_path=joinpath(workspace, "notebook_helpers", "julia", "plot_settings.toml")
shared_plot_settings=TOML.parsefile(shared_plot_settings_path)
beam_color_min=Float64(shared_plot_settings["beam_intensity_min_W_per_um2"])
beam_color_max=Float64(shared_plot_settings["beam_intensity_max_W_per_um2"])
beam_colormap=shared_plot_settings["colormap"]
beam_color_ticks=collect(range(beam_color_min,beam_color_max;length=6))
@printf("Largest displayed physical intensity: %.9f W/μm²; shared scale %.3f to %.3f W/μm²\n",
    observed_max,beam_color_min,beam_color_max)
write_toml(joinpath(run_dir,"beam_plot_settings.toml"),Dict(
    "shared_min_W_per_um2"=>beam_color_min,"shared_max_W_per_um2"=>beam_color_max,
    "observed_max_W_per_um2"=>observed_max,"shared_settings_source"=>shared_plot_settings_path))
for P in power_cases_W
    fig,axes=subplots(2,3;figsize=(15,8.6),constrained_layout=true)
    meshes=[]
    for (i,(label,row)) in enumerate((("baseline",validations[P].base_fine),("selected",validations[P].best_fine)))
        maps=beam_maps[(P,label)]
        for j in 1:3
            ax=axes[i,j];m=maps[j]
            mesh=ax.pcolormesh(x.*1e6,y.*1e6,m.intensity;cmap=beam_colormap,vmin=beam_color_min,vmax=beam_color_max,
                shading="auto",rasterized=true)
            push!(meshes,mesh)
            ax.set_title(@sprintf("%s: %s\npeak %.2f kW at %.3f ps",label,curve_labels[j],m.peak_power_W/1000,m.peak_time_ps);fontsize=10)
            ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)");ax.set_aspect("equal")
        end
        npzwrite(joinpath(case_dir(P),label*"_beam_maps.npz"),Dict(
            "x_um"=>x.*1e6,"y_um"=>y.*1e6,
            "input_W_per_um2"=>maps[1].intensity,"linear_W_per_um2"=>maps[2].intensity,
            "nonlinear_W_per_um2"=>maps[3].intensity,
            "peak_times_ps"=>[m.peak_time_ps for m in maps],"peak_powers_W"=>[m.peak_power_W for m in maps],
            "shared_color_max_W_per_um2"=>beam_color_max,"launch_coefficients"=>coefficients(row)))
    end
    fig.colorbar(meshes[end];ax=vec(axes),shrink=0.8,ticks=beam_color_ticks,
        label="Intensity at each pulse's peak (W/μm²)")
    fig.suptitle(@sprintf("%.3f kW: coherent whole beam | start %s | selected %s",
        P/1000,coeff_label(start_coefficients),coeff_label(coefficients(validations[P].best_fine)));fontsize=11)
    finish_figure(fig,joinpath(case_dir(P),"baseline_selected_beams"))
end
println("Spatial integrals match summed modal peak powers for all displayed whole-beam maps.")

In [ ]:
for P in power_cases_W
    fig,axes=subplots(2,3;figsize=(15,8.6),constrained_layout=true)
    meshes=[]
    for m in filter(r->r.power_W==P,individual_maps)
        ax=axes[m.case_index,m.mode]
        mesh=ax.pcolormesh(x.*1e6,y.*1e6,m.intensity;cmap=beam_colormap,vmin=beam_color_min,vmax=beam_color_max,
            shading="auto",rasterized=true)
        push!(meshes,mesh)
        ax.set_title(@sprintf("%s: %s\nenergy %.3g%% | peak %.3g kW at %.3f ps",
            m.label,mode_names[m.mode],100m.energy_fraction,m.peak_power_W/1000,m.time_ps);fontsize=10)
        ax.set_xlabel("x (μm)");ax.set_ylabel("y (μm)");ax.set_aspect("equal")
    end
    fig.colorbar(meshes[end];ax=vec(axes),shrink=0.8,ticks=beam_color_ticks,
        label="Single-mode peak intensity (W/μm²)")
    fig.suptitle(@sprintf("%.3f kW: individual nonlinear output modes",P/1000))
    finish_figure(fig,joinpath(case_dir(P),"baseline_selected_output_modes"))
end

## 12. Read the bounded-search result

`verified_improvement` requires a lower 1 mm objective by more than the stated tolerance
and passing core checks. A small projected gradient applies only to the search grid.
An iteration limit, evaluation limit, or failed line search is recorded as the actual
stop reason. No fine-grid stationarity or global minimum is established by this run.

CSV files record evaluations, rejected line-search candidates, accepted states, and
baseline/selected comparisons. TOML files store settings, checks and stop reasons.
NPZ files hold the displayed fields. The objective covers mode 1 only; all three modes propagate with equal
launch phases, and the brightness floor counts LP01 photons; both fixed-power searches begin again at equal amplitudes.

In [ ]:
for P in power_cases_W
    s=summaries[P]
    println("\n",power_label(P))
    println("  coefficients: ",coeff_label(s["selected_coefficients"]))
    fractions=s["selected_input_power_fractions"]
    @printf("  input power fractions: [%.3f%%, %.3f%%, %.3f%%]\n",100fractions[1],100fractions[2],100fractions[3])
    @printf("  LP01 fine D_IO: %.8f -> %.8f (%.2f%% reduction)\n",
        s["baseline_D_IO_fine"],s["selected_D_IO_fine"],100s["fractional_D_IO_reduction"])
    println("  stop: ",s["stop_reason"],"; accepted steps: ",s["accepted_steps"])
    println("  core checks passed: ",s["core_checks_passed"],
        "; verified improvement: ",s["verified_improvement"])
    failed_checks=sort([name for (name,status) in s["checks"] if status=="fail"])
    isempty(failed_checks) || println("  Failed checks: ",join(failed_checks,", "))
    @printf("  max baseline/selected 5 mm vs 1 mm D_IO difference: %.3g (tolerance %.3g)\n",
        s["max_step_size_D_IO_difference"],dio_tolerance)
    @printf("  new evaluations: optimization %d/%d; baseline + selected checks %d/%d\n",
        used[P][:optimization],max_optimizer_evaluations,used[P][:validation],max_check_evaluations)
    println("  Fine-grid stationarity and a global minimum are not established.")
end
@printf("\nMaximum displayed physical intensity: %.9f W/μm²\n",observed_max)
println("Results: ",run_dir)